In [0]:
import requests
import json
import logging
import base64
from datetime import datetime, timezone, timedelta

# ============================================================
# 1. SETUP & CONFIGURATION
# ============================================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ============================================================
# 2. CONFIGURATION — Update these
# ============================================================
TABLE_NAME = "courseify.default.jobs_bronze"
REPO_OWNER = "courseify-jobs-1"          # Your GitHub Username
REPO_NAME  = "jobs_at_courseify"         # YOUR NEW REPO NAME
BRANCH     = "main"                      
MAX_JOBS   = 30

IST = timezone(timedelta(hours=5, minutes=30))

try:
    GROQ_API_KEY = dbutils.secrets.get("courseify", "groq_api_key")
    GITHUB_TOKEN = dbutils.secrets.get("courseify", "github-token-daily-commit")
except Exception as e:
    logger.error("❌ Secrets error.")
    raise e

# ============================================================
# 2. FETCH JOBS
# ============================================================
def fetch_jobs_from_db():
    logger.info("Fetching latest jobs...")
    try:
        df = spark.sql(f"""
            SELECT title, company, location, type, salary, category, apply_link
            FROM {TABLE_NAME}
            ORDER BY inserted_at DESC
            LIMIT {MAX_JOBS}
        """)
        jobs = [row.asDict() for row in df.collect()]
        for job in jobs:
            for k, v in job.items():
                if v is None: job[k] = 'Not specified'
        return jobs
    except Exception as e:
        logger.error(f"Failed to fetch jobs: {e}")
        return []

# ============================================================
# 3. GENERATE DATA-DRIVEN INSIGHT (USING GROQ)
# ============================================================
def generate_insight(jobs):
    logger.info("Generating data-driven insight via Groq...")
    
    # Extract real data to feed the prompt
    titles = [j['title'] for j in jobs[:10]]
    companies = list(set([j['company'] for j in jobs[:15]]))
    
    prompt = f"""
    You are a Data Analyst for an Indian Tech Job portal.
    Analyze this sample of today's job postings:
    Titles: {", ".join(titles)}
    Companies: {", ".join(companies)}
    
    Write a 150-word daily market insight article based ONLY on these trends. 
    Provide actionable advice for job seekers.
    Return strictly as JSON:
    {{
        "title": "Catchy Headline including India and Month/Year",
        "excerpt": "A 1-sentence summary",
        "content": "The full 150 word article with paragraphs separated by \\n",
        "tags": ["tag1", "tag2"]
    }}
    """
    
    try:
        res = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
            json={
                "model": "llama-3.3-70b-versatile",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.5,
                "response_format": {"type": "json_object"}
            }
        )
        res.raise_for_status()
        raw = res.json()["choices"][0]["message"]["content"]
        insight = json.loads(raw)
        
        # Add metadata
        now_ist = datetime.now(IST)
        insight['date'] = now_ist.strftime("%B %d, %Y")
        insight['id'] = now_ist.strftime("%Y%m%d%H%M")
        return insight
    except Exception as e:
        logger.error(f"Groq generation failed: {e}")
        return None

# ============================================================
# 4. GITHUB REPO MANAGEMENT
# ============================================================
def get_file_from_github(path):
    url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/{path}?ref={BRANCH}"
    headers = {"Authorization": f"token {GITHUB_TOKEN}"}
    res = requests.get(url, headers=headers)
    if res.status_code == 200:
        data = res.json()
        content = base64.b64decode(data['content']).decode('utf-8')
        return content, data['sha']
    return None, None

def push_to_github(path, content, message, sha=None):
    url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/{path}"
    headers = {"Authorization": f"token {GITHUB_TOKEN}"}
    payload = {
        "message": message,
        "content": base64.b64encode(content.encode('utf-8')).decode('utf-8'),
        "branch": BRANCH
    }
    if sha: payload["sha"] = sha
    
    res = requests.put(url, headers=headers, json=payload)
    if res.status_code in [200, 201]:
        logger.info(f"✅ Successfully pushed: {path}")
    else:
        logger.error(f"❌ Failed to push {path}: {res.text}")

# ============================================================
# 5. GENERATE RSS FEED FOR GOOGLE NEWS
# ============================================================
def generate_rss(insights):
    pub_date = datetime.now(timezone.utc).strftime("%a, %d %b %Y %H:%M:%S GMT")
    site_url = f"https://{REPO_OWNER}.github.io/{REPO_NAME}/"
    
    items_xml = ""
    # Add top 10 insights to RSS feed
    for post in insights[:10]:
        # Convert YYYYMMDDHHMM to a proper RSS pubDate (rough approx for past dates)
        try:
            post_dt = datetime.strptime(post['id'], "%Y%m%d%H%M").replace(tzinfo=IST)
            item_pub_date = post_dt.astimezone(timezone.utc).strftime("%a, %d %b %Y %H:%M:%S GMT")
        except:
            item_pub_date = pub_date

        items_xml += f"""
        <item>
            <title><![CDATA[{post['title']}]]></title>
            <link>{site_url}insights.html#{post['id']}</link>
            <guid isPermaLink="false">{post['id']}</guid>
            <pubDate>{item_pub_date}</pubDate>
            <description><![CDATA[{post['excerpt']}]]></description>
            <content:encoded><![CDATA[<p>{post['content'].replace(chr(10), '</p><p>')}</p>]]></content:encoded>
        </item>"""

    rss = f"""<?xml version="1.0" encoding="UTF-8" ?>
<rss version="2.0" xmlns:atom="http://www.w3.org/2005/Atom" xmlns:content="http://purl.org/rss/1.0/modules/content/">
<channel>
    <title>Courseify Daily Insights</title>
    <link>{site_url}</link>
    <description>Data-driven market insights for Indian Tech Professionals.</description>
    <language>en-in</language>
    <pubDate>{pub_date}</pubDate>
    <lastBuildDate>{pub_date}</lastBuildDate>
    <atom:link href="{site_url}feed.xml" rel="self" type="application/rss+xml" />
    {items_xml}
</channel>
</rss>"""
    return rss

# ============================================================
# 6. MAIN EXECUTION FLOW
# ============================================================
def run_publisher():
    jobs = fetch_jobs_from_db()
    if not jobs: return

    insight = generate_insight(jobs)
    
    # 1. Update Jobs JSON
    _, jobs_sha = get_file_from_github("data/latest_jobs.json")
    push_to_github("data/latest_jobs.json", json.dumps(jobs, indent=2), "Auto: Update Jobs", jobs_sha)

    # 2. Update Insights JSON (Append to history)
    if insight:
        existing_insights_str, insights_sha = get_file_from_github("data/insights.json")
        try:
            insights_list = json.loads(existing_insights_str) if existing_insights_str else []
        except:
            insights_list = []
            
        insights_list.insert(0, insight) # Add new insight to the top
        push_to_github("data/insights.json", json.dumps(insights_list, indent=2), "Auto: Add Daily Insight", insights_sha)
        
        # 3. Update RSS Feed
        rss_content = generate_rss(insights_list)
        _, rss_sha = get_file_from_github("feed.xml")
        push_to_github("feed.xml", rss_content, "Auto: Update RSS Feed", rss_sha)

if __name__ == "__main__":
    run_publisher()